# 09 · Nhập Music4All-Onion (last.fm) lên Kaggle (ticket #54)

Bộ dữ liệu thứ hai cho phần điểm mix (mở rộng, #54–#57): lịch sử nghe last.fm và đặc trưng Essentia. Notebook tải thẳng từ Zenodo về `/tmp` của Kaggle, kiểm md5, đổi sang parquet; **không tải về máy cá nhân**.

**Nguồn (CC BY 4.0):** M. Moscati, E. Parada-Cabaleiro, Y. Deldjoo, E. Zangerle, M. Schedl. *Music4All-Onion: A Large-Scale Multi-faceted Content-Centric Music Recommendation Dataset*. CIKM 2022. Dữ liệu: Zenodo record 15394646, DOI [10.5281/zenodo.15394646](https://doi.org/10.5281/zenodo.15394646). Nhóm chỉ đổi định dạng và chọn cột; không sửa giá trị.

**Input:** không cần Dataset; bật **Internet**. Tải `userid_trackid_timestamp.tsv.bz2` (2,2 GB), `id_essentia.tsv.bz2` (0,4 GB), `README.md`.

**Output** (`/kaggle/working/out/onion/`):
- `listening_events.parquet`: `user_id` int32, `track_id` string (16 ký tự, không phải Spotify ID), `timestamp` (giây); mọi dòng, chưa lọc
- `essentia.parquet`: `id` và 46 cột đặc trưng (BPM và hai đỉnh histogram BPM, onset rate, danceability, độ mạnh tông theo 3 phương pháp, loudness, 36 bin HPCP trung bình)
- `import_log.json`: md5, số dòng, số user, số bài, khoảng thời gian, đối chiếu với README của record
- `README.md` bản gốc từ Zenodo

Chỉ cần CPU. Ước tính 30–60 phút (giải nén bz2 chạy một luồng). Repo clone vào `/tmp`, file gốc tải vào `/tmp/onion`, nên output chỉ có `out/`.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #54 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
# Dung lượng: file gốc khoảng 2,6 GB ở /tmp; output parquet ở /kaggle/working (giới hạn 20 GB)
import shutil
for path in ("/tmp", "/kaggle/working"):
    print(path, f"còn trống {shutil.disk_usage(path).free / 1e9:.1f} GB")

In [ ]:
import json, time
from src.data.onion import main

t0 = time.time()
log = main(["--work", "/tmp/onion", "--out", "/kaggle/working/out/onion"])   # tải, kiểm md5, đổi sang parquet
print(f"{time.time() - t0:.0f} giây")

In [ ]:
# Đối chiếu với README của record và xem nhanh dữ liệu
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

for key, c in log["check"].items():
    print(f"{key:16s} kỳ vọng {c['expected']:>12,} | thực tế {c['actual']:>12,} | {'khớp' if c['ok'] else 'LỆCH'}")
OUT = Path("/kaggle/working/out/onion")
for p in sorted(OUT.iterdir()):
    print(p.name, f"{p.stat().st_size / 1e6:.1f} MB")
print(pq.ParquetFile(OUT / "listening_events.parquet").read_row_group(0).to_pandas().head())
print(pd.read_parquet(OUT / "essentia.parquet").iloc[:3, :8])

## Đăng output

1. **Save Version → Save & Run All (Commit)**.
2. Tab Output → **New Dataset** tên **`mixnext-onion`**, **Visibility: Private**, License: CC BY 4.0 (theo bản gốc). Ghi nguồn như ô đầu notebook vào mô tả Dataset.
3. Notebook 10 (#55, #56) Add Input bộ này và tìm file bằng `Path("/kaggle/input").rglob(...)`.

## Kết quả (Kaggle CPU, 27/11/2026, 1.888 giây)

| | kỳ vọng (README của record) | thực tế | |
|---|---:|---:|---|
| lượt nghe | 252.984.396 | 252.984.396 | khớp |
| user | 119.140 | 119.140 | khớp |
| bài có lượt nghe | 56.512 | 56.512 | khớp |
| bài trong `essentia.parquet` | (README để trống; các bộ đặc trưng khác 109.269) | 109.180 | ghi lại |

- md5 của `userid_trackid_timestamp.tsv.bz2` và `id_essentia.tsv.bz2` khớp Zenodo.
- `essentia.parquet` có 109.180 bài, 47 cột (không trùng id), 30,2 MB; `listening_events.parquet` 2.269,8 MB.
- `timestamp` từ **1970-01-01 01:00:36** đến 2020-03-20 12:59:51. Mốc 1970 là epoch 0 của Unix time: lượt nghe thiếu thời điểm bị ghi thành 0. **#55 phải lọc các lượt này** trước khi cắt phiên, nếu không sẽ thành khoảng nghỉ khổng lồ hoặc phiên giả.
- Mọi bài có lượt nghe (56.512) đều nằm trong danh mục Music4All; tỉ lệ có đặc trưng Essentia đo ở #55.